In [1]:
from warnings import simplefilter
simplefilter('ignore')

import numpy as np
from llama_cpp import Llama

In [2]:
# Download the model from Hugging Face on the first run and load it locally.
# Later runs normally reuse the model stored in the Hugging Face cache.
model = Llama.from_pretrained(
    repo_id="Qwen/Qwen3-4B-GGUF",
    filename="Qwen3-4B-Q8_0.gguf",
    # The maximum prompt size is 512 tokens
    n_ctx=512,
    # Retain raw vocabulary scores so we can inspect the final prompt position.
    logits_all=True,
    # Suppress llama.cpp diagnostic messages.
    verbose=False,
)

In [3]:
choices = ["Good", "Bad"]

labels = np.arange(1, len(choices) + 1).astype(str)

query = "Perfectly cooked, assuming the goal was disappointment"

system_prompt = """
You are a food review sentiment classifier. Analyze the given food review and classify its intended sentiment as exactly one of:

["Good", "Bad"]

Classification rules:
- Good: The reviewer ultimately expresses a positive opinion about the food.
- Bad: The reviewer ultimately expresses a negative opinion about the food.
- Consider the intended meaning of the review, not just the literal words.
- Pay special attention to sarcasm, irony, exaggeration, contradictions, and context.
- If positive words are used sarcastically to express dissatisfaction, classify the review as "Bad".
- If negative words are used sarcastically to express satisfaction, classify the review as "Good".
"""

options = "\n".join(
    f"{label}. {choice}" for label, choice in zip(labels, choices)
)

prompt = f"""<|im_start|>system
{system_prompt}\n\nChoose one option.<|im_end|>
<|im_start|>user
{query}\n\n{options}<|im_end|>
<|im_start|>assistant
<think>\n\n</think>\n\n"""

print(prompt)

<|im_start|>system

You are a food review sentiment classifier. Analyze the given food review and classify its intended sentiment as exactly one of:

["Good", "Bad"]

Classification rules:
- Good: The reviewer ultimately expresses a positive opinion about the food.
- Bad: The reviewer ultimately expresses a negative opinion about the food.
- Consider the intended meaning of the review, not just the literal words.
- Pay special attention to sarcasm, irony, exaggeration, contradictions, and context.
- If positive words are used sarcastically to express dissatisfaction, classify the review as "Bad".
- If negative words are used sarcastically to express satisfaction, classify the review as "Good".


Choose one option.<|im_end|>
<|im_start|>user
Perfectly cooked, assuming the goal was disappointment

1. Good
2. Bad<|im_end|>
<|im_start|>assistant
<think>

</think>




In [4]:
# Convert the prompt from text to token IDs, then run one forward evaluation.
# add_bos=False avoids adding an extra beginning token because the chat prompt
# already has its own structure. special=True recognizes tokens such as
# <|im_start|> instead of treating them as ordinary text. No text is generated.
model.eval(model.tokenize(prompt.encode(), add_bos=False, special=True))

# Get the vocabulary-wide next-token scores after the last prompt token.
# model.n_tokens - 1 is the last position because Python indexes from zero.
logits = model.scores[model.n_tokens - 1]

# Translate A, B, and C into their vocabulary token IDs. [0] assumes each
# label is exactly one token, which is true for these labels with this model.
token_ids = [model.tokenize(label.encode(), add_bos=False)[0] for label in labels]

# Ignore the rest of the vocabulary and keep only the scores for A, B, and C.
choice_logits = np.asarray([logits[token_id] for token_id in token_ids])

# Apply a numerically stable softmax. logaddexp.reduce computes log(sum(exp(x))).
# The resulting values sum to 1 across these three choices. They are conditional
# choice probabilities, not necessarily calibrated real-world confidence scores.
probabilities = np.exp(choice_logits - np.logaddexp.reduce(choice_logits))

# Match each probability back to its human-readable choice and print it.
# :10 aligns the names; .1% formats 0.88 as 88.0%.
for choice, probability in zip(choices, probabilities, strict=True):
    print(f"{choice:10} {probability:.1%}")


Good       0.0%
Bad        100.0%
